# 02 · Exploratory data analysis
**AAI-540 · Group 4 · Criteo CTR**

Runs on the curated chronological sample from notebook 01, with full-dataset cross-checks through Athena.
Every finding in §12 is **computed**, not hand-written, and saved to `artifacts/eda_summary.json` so the design
document and check-ins can cite the exact numbers.

Questions this notebook answers, each tied to a design decision:

| § | Question | Decision it informs |
|---|---|---|
| 2 | How imbalanced is the label? | `scale_pos_weight`, metric choice |
| 3 | Is CTR stable over the 7 days? | chronological split, drift monitoring |
| 4–5 | How much is missing, and is missingness predictive? | missing-indicator features |
| 6–7 | How skewed/redundant are numeric features? Any negatives? | signed-log transform, feature pruning |
| 8 | Which features carry signal? | tests the RFC hypothesis that a few categoricals dominate |
| 9–11 | How extreme is categorical cardinality, and how many levels are unseen later? | hashing + rare bucketing, unseen-level monitoring |

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

from criteo_ctr import config as C
from criteo_ctr import features as F
from criteo_ctr.io_utils import Store, s3_uri

LOCAL = bool(os.environ.get("CRITEO_LOCAL_ROOT"))   # offline testing only
if LOCAL:
    bucket, athena = "local", None
    store = Store(bucket)
else:
    import sagemaker
    from criteo_ctr.athena import Athena
    sess = sagemaker.Session()
    bucket = C.BUCKET or sess.default_bucket()
    store = Store(bucket, sess.boto_session)
    athena = Athena(sess.boto_session, s3_uri(bucket, C.ATHENA_RESULTS_PREFIX) + "/", database=C.ATHENA_DATABASE)

FIG_DIR = os.path.abspath("../reports/figures"); os.makedirs(FIG_DIR, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, bbox_inches="tight")
    if not LOCAL:
        store.upload_file(path, f"{C.PREFIX}/reports/figures/{name}")

summary = {}   # everything computed here lands in eda_summary.json

## 1 · Load the curated sample

In [ ]:
sample = store.read_parquet_prefix(C.SAMPLE_PREFIX)
sample = sample.astype({**{c: "Int64" for c in C.NUM_COLS}, **{c: "string" for c in C.CAT_COLS},
                        C.LABEL: "int64", C.DAY_INDEX: "int64"})
sample = sample.sort_values(C.RECORD_ID).reset_index(drop=True)
LABEL = C.LABEL
print(f"{len(sample):,} rows x {sample.shape[1]} columns | "
      f"{sample.memory_usage(deep=True).sum()/1e6:,.0f} MB in memory")
summary["sample_rows"] = int(len(sample))
sample.head(3)

## 2 · Label balance

In [ ]:
ctr = sample[LABEL].mean()
summary["sample_ctr"] = float(ctr)
summary["neg_per_pos"] = float((1 - ctr) / ctr)
print(f"sample CTR = {ctr:.4%}  ->  {summary['neg_per_pos']:.2f} negatives per positive")

if athena:
    full = athena.query(f"SELECT count(*) n, avg(label) ctr FROM {C.RAW_TABLE}")
    summary["full_rows"], summary["full_ctr"] = int(full.n[0]), float(full.ctr[0])
    print(f"full data : {summary['full_rows']:,} rows, CTR = {summary['full_ctr']:.4%}")

fig, ax = plt.subplots(figsize=(5, 3.2))
sample[LABEL].value_counts().sort_index().rename({0: "no click (0)", 1: "click (1)"}).plot.bar(ax=ax, color=["#8da0cb", "#fc8d62"])
ax.set_title(f"Label distribution (CTR = {ctr:.2%})"); ax.set_ylabel("impressions"); ax.tick_params(axis="x", rotation=0)
savefig("01_label_balance.png"); plt.show()

> **Interpretation guide.** Criteo subsampled negatives when releasing this dataset, so the click rate here is far
> higher than real-world display CTR (typically well under 1%). If the measured CTR is in the ~20–30% range,
> the class imbalance is *moderate* rather than extreme, and the design document's "heavily imbalanced" wording and
> the `scale_pos_weight` rationale should be updated to match. Predicted probabilities would also need
> recalibrating to the true base rate before use in bidding.

## 3 · CTR over time

In [ ]:
by_day = sample.groupby(C.DAY_INDEX)[LABEL].agg(rows="size", ctr="mean")
bins = pd.cut(sample[C.RECORD_ID], bins=7 * 24, labels=False)          # ~hourly if traffic were uniform
by_bin = sample.groupby(bins)[LABEL].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1, 2]})
by_day["ctr"].plot(ax=axes[0], marker="o"); axes[0].axhline(ctr, ls="--", c="grey")
axes[0].set_title("CTR by day_index"); axes[0].set_xlabel("day"); axes[0].set_ylabel("CTR")
by_bin.plot(ax=axes[1], lw=0.9); by_bin.rolling(12, center=True).mean().plot(ax=axes[1], lw=2)
axes[1].set_title("CTR by position bin (168 bins ≈ hourly)"); axes[1].set_xlabel("position bin"); axes[1].set_ylabel("CTR")
for d in range(1, 7):
    axes[1].axvline(d * 24, c="grey", lw=0.5, ls=":")
savefig("02_ctr_over_time.png"); plt.show()

summary["ctr_by_day"] = {int(k): float(v) for k, v in by_day["ctr"].items()}
summary["ctr_day_range_pp"] = float((by_day["ctr"].max() - by_day["ctr"].min()) * 100)
by_day

## 4 · Missing values

In [ ]:
miss = sample[C.NUM_COLS + C.CAT_COLS].isna().mean().sort_values(ascending=False)
colors = ["#66c2a5" if c.startswith("I") else "#8da0cb" for c in miss.index]
fig, ax = plt.subplots(figsize=(13, 3.6))
ax.bar(miss.index, miss.values, color=colors)
ax.set_title("Missing-value rate per feature (green = numeric, blue = categorical)"); ax.set_ylabel("share missing")
ax.tick_params(axis="x", rotation=90)
savefig("03_missing_rates.png"); plt.show()

summary["missing_rate"] = {k: float(v) for k, v in miss.items()}
summary["features_over_40pct_missing"] = miss[miss > 0.40].index.tolist()
summary["features_complete"] = miss[miss == 0].index.tolist()

if athena:   # one pass over the full raw table for all 39 columns
    exprs = ", ".join(f"avg(CASE WHEN {c.lower()} IS NULL THEN 1.0 ELSE 0.0 END) AS {c.lower()}"
                      for c in C.NUM_COLS + C.CAT_COLS)
    full_miss = athena.query(f"SELECT {exprs} FROM {C.RAW_TABLE}").iloc[0]
    full_miss.index = [i.upper() for i in full_miss.index]
    gap = (full_miss - miss.reindex(full_miss.index)).abs().max()
    summary["missing_rate_max_abs_gap_sample_vs_full"] = float(gap)
    print(f"largest sample-vs-full missing-rate gap: {gap:.4f}")
miss.to_frame("missing_rate").T

## 5 · Is missingness itself predictive?

In [ ]:
rows = []
for c in C.NUM_COLS + C.CAT_COLS:
    m = sample[c].isna()
    if 0 < m.mean() < 1:
        rows.append({"feature": c, "missing_rate": m.mean(),
                     "ctr_when_missing": sample.loc[m, LABEL].mean(),
                     "ctr_when_present": sample.loc[~m, LABEL].mean()})
mv = pd.DataFrame(rows)
mv["ctr_ratio"] = mv.ctr_when_missing / mv.ctr_when_present
mv = mv.reindex(mv.ctr_ratio.sub(1).abs().sort_values(ascending=False).index).reset_index(drop=True)

top = mv.head(12)
fig, ax = plt.subplots(figsize=(10, 3.6))
x = np.arange(len(top)); w = 0.4
ax.bar(x - w/2, top.ctr_when_missing, w, label="missing"); ax.bar(x + w/2, top.ctr_when_present, w, label="present")
ax.set_xticks(x, top.feature); ax.set_ylabel("CTR"); ax.legend(); ax.set_title("CTR when the feature is missing vs present")
savefig("04_missingness_vs_ctr.png"); plt.show()

informative = mv[(mv.ctr_ratio < 0.9) | (mv.ctr_ratio > 1.1)]
summary["informative_missingness"] = informative.feature.tolist()
print(f"{len(informative)} features where missing shifts CTR by >10% relative -> keep explicit missing flags")
mv.round(4).head(15)

## 6 · Numeric distributions

In [ ]:
num = sample[C.NUM_COLS].astype("float64")
desc = num.describe(percentiles=[.5, .95, .99]).T
desc["skew"] = num.skew()
desc["share_zero"] = (num == 0).sum() / num.notna().sum()
desc["n_negative"] = (num < 0).sum()
summary["numeric_skew"] = desc["skew"].round(2).to_dict()
summary["numeric_with_negatives"] = desc.index[desc.n_negative > 0].tolist()
print("features with negative values (plain log1p would fail):", summary["numeric_with_negatives"])

fig, axes = plt.subplots(3, 5, figsize=(15, 7.5)); axes = axes.ravel()
for ax, c in zip(axes, C.NUM_COLS):
    v = F.signed_log1p(sample[c]).dropna()
    ax.hist(v, bins=50, color="#66c2a5"); ax.set_title(f"{c}  (raw skew {desc.loc[c, 'skew']:.1f})", fontsize=9)
for ax in axes[len(C.NUM_COLS):]:
    ax.axis("off")
fig.suptitle("Numeric features after signed log1p transform", y=1.0)
plt.tight_layout(); savefig("05_numeric_distributions.png"); plt.show()
desc.round(2)

## 7 · Numeric redundancy

In [ ]:
corr = num.corr(method="spearman")
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, annot=True, fmt=".1f", annot_kws={"size": 7}, ax=ax)
ax.set_title("Spearman correlation - numeric features"); savefig("06_numeric_correlation.png"); plt.show()

pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(C.NUM_COLS) for b in C.NUM_COLS[i + 1:]
         if abs(corr.loc[a, b]) >= 0.7]
summary["highly_correlated_numeric_pairs"] = [[a, b, round(float(r), 3)] for a, b, r in pairs]
print("pairs with |rho| >= 0.7:", summary["highly_correlated_numeric_pairs"] or "none")

## 8 · Which features carry signal?
Univariate AUC per feature, reported as `max(AUC, 1−AUC)` so direction does not matter (0.5 = no signal).
For categoricals, a level-CTR table is **fitted on the first half of the sample and scored on the second half**
(chronologically) — fitting and scoring on the same rows would reward memorising rare levels and inflate the result.

In [ ]:
half = len(sample) // 2
early, late = sample.iloc[:half], sample.iloc[half:]
y_late = late[LABEL].to_numpy()
prior = early[LABEL].mean()
rows = []
for c in C.NUM_COLS:
    x = F.signed_log1p(late[c]).fillna(-1).to_numpy()
    a = roc_auc_score(y_late, x)
    rows.append({"feature": c, "type": "numeric", "auc": max(a, 1 - a)})
for c in C.CAT_COLS:
    g = early.groupby(early[c].fillna("__MISSING__"))[LABEL].agg(["sum", "count"])
    level_ctr = (g["sum"] + 20 * prior) / (g["count"] + 20)          # smoothed toward the prior
    x = late[c].fillna("__MISSING__").map(level_ctr).fillna(prior).astype("float64").to_numpy()
    a = roc_auc_score(y_late, x)
    rows.append({"feature": c, "type": "categorical", "auc": max(a, 1 - a)})
signal = pd.DataFrame(rows).sort_values("auc", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.bar(signal.feature, signal.auc - 0.5, bottom=0.5,
       color=["#66c2a5" if t == "numeric" else "#8da0cb" for t in signal.type])
ax.axhline(0.5, c="grey", lw=0.8); ax.set_ylabel("univariate AUC"); ax.tick_params(axis="x", rotation=90)
ax.set_title("Univariate signal strength, early half -> late half (green numeric, blue categorical)")
savefig("07_univariate_signal.png"); plt.show()

top10 = signal.head(10)
summary["univariate_auc"] = signal.set_index("feature").auc.round(4).to_dict()
summary["top10_features"] = top10.feature.tolist()
summary["top10_categorical_share"] = float((top10.type == "categorical").mean())
print(f"top 10 by signal: {top10.feature.tolist()}  ({summary['top10_categorical_share']:.0%} categorical)")
signal.head(15).round(4)

## 9 · Categorical cardinality

In [ ]:
card_rows = []
for c in C.CAT_COLS:
    vc = sample[c].value_counts()
    card_rows.append({"feature": c, "levels_in_sample": len(vc),
                      "top1_share": vc.iloc[0] / sample[c].notna().sum() if len(vc) else np.nan,
                      "share_levels_seen_once": (vc == 1).mean(),
                      "share_rows_in_levels_lt_10": vc[vc < 10].sum() / vc.sum()})
card = pd.DataFrame(card_rows).set_index("feature").sort_values("levels_in_sample", ascending=False)

if athena:
    exprs = ", ".join(f"approx_distinct({c.lower()}) AS {c.lower()}" for c in C.CAT_COLS)
    full_card = athena.query(f"SELECT {exprs} FROM {C.RAW_TABLE}").iloc[0]
    card["levels_full_data_approx"] = [int(full_card[c.lower()]) for c in card.index]

fig, ax = plt.subplots(figsize=(12, 3.6))
col = "levels_full_data_approx" if "levels_full_data_approx" in card else "levels_in_sample"
ax.bar(card.index, card[col], color="#8da0cb"); ax.set_yscale("log")
ax.set_title(f"Categorical cardinality ({'full data, approx' if col != 'levels_in_sample' else 'sample'}; log scale)")
ax.tick_params(axis="x", rotation=90); savefig("08_categorical_cardinality.png"); plt.show()

summary["cardinality"] = card[col].astype(int).to_dict()
summary["cardinality_source"] = col
summary["cat_cols_over_100k_levels"] = card.index[card[col] > 100_000].tolist()
card.round(3)

## 10 · CTR across the top levels of the strongest categorical features

In [ ]:
top_cats = signal[signal.type == "categorical"].feature.head(3).tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, c in zip(axes, top_cats):
    g = sample.groupby(sample[c].fillna("__MISSING__"))[LABEL].agg(rows="size", ctr="mean")
    g = g.sort_values("rows", ascending=False).head(10)
    ax.bar(range(len(g)), g.ctr, color="#fc8d62"); ax.axhline(ctr, c="grey", ls="--", lw=0.8)
    ax.set_xticks(range(len(g)), [s[:6] for s in g.index], rotation=60, fontsize=8)
    ax.set_title(f"{c}: CTR of 10 most frequent levels", fontsize=10); ax.set_ylabel("CTR")
plt.tight_layout(); savefig("09_top_level_ctr.png"); plt.show()

## 11 · Unseen categories after the training window
Share of **production-period rows** (last 40% by time) whose level never appeared in the **training period**
(first 40%). This is exactly what the model will face after deployment, so it sizes the rare/unseen bucket and
sets the baseline for the "unseen-category rate" monitor in the design document.

In [ ]:
split = F.assign_chronological_split(len(sample))
train_p, prod_p = sample[split == "train"], sample[split == "production"]
unseen = {}
for c in C.CAT_COLS:
    seen = set(train_p[c].dropna().unique())
    present = prod_p[c].dropna()
    unseen[c] = float((~present.isin(seen)).mean()) if len(present) else 0.0
unseen = pd.Series(unseen).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 3.4))
ax.bar(unseen.index, unseen.values, color="#e78ac3"); ax.tick_params(axis="x", rotation=90)
ax.set_ylabel("share of rows"); ax.set_title("Production-period rows with a level unseen in the training period")
savefig("10_unseen_levels.png"); plt.show()
summary["unseen_rate_production_vs_train"] = unseen.round(4).to_dict()
summary["cat_cols_unseen_over_10pct"] = unseen[unseen > 0.10].index.tolist()
unseen.to_frame("unseen_rate").T.round(3)

## 12 · Computed findings → `eda_summary.json`

In [ ]:
lines = []
full_ctr = summary.get("full_ctr", summary["sample_ctr"])
lines.append(f"Label: CTR {full_ctr:.2%} ({summary['neg_per_pos']:.1f} negatives per positive) - "
             + ("moderate imbalance (negatives were subsampled by Criteo)." if full_ctr > 0.1 else "heavy imbalance."))
lines.append(f"Time: daily CTR varies by {summary['ctr_day_range_pp']:.2f} percentage points across the 7 days.")
n_inf = len(summary["informative_missingness"])
lines.append(f"Missingness: {len(summary['features_over_40pct_missing'])} features are >40% missing "
             f"({', '.join(summary['features_over_40pct_missing']) or 'none'}). Missingness shifts CTR by >10% "
             f"for {n_inf} features" + (" - supports keeping explicit missing flags." if n_inf
                                        else " - missing flags are low-value; consider dropping them."))
neg, pairs = summary["numeric_with_negatives"], summary["highly_correlated_numeric_pairs"]
lines.append("Numeric: " + (f"negative values in {neg} - signed log1p required (plain log1p would fail). "
                            if neg else "no negative values. ")
             + (f"{len(pairs)} numeric pairs with |rho| >= 0.7 - candidates for pruning." if pairs
                else "no numeric pairs with |rho| >= 0.7."))
share = summary["top10_categorical_share"]
lines.append(f"Signal: top 10 univariate features are {share:.0%} categorical ({', '.join(summary['top10_features'])}) - "
             + ("consistent with" if share >= 0.6 else "does NOT support") + " the RFC hypothesis that categoricals dominate.")
big = summary["cat_cols_over_100k_levels"]
lines.append(f"Cardinality: {len(big)} categorical columns exceed 100k levels ({', '.join(big) or 'none'})"
             + (" - hashing to a fixed space is required." if big else " - one-hot encoding may be feasible for some columns."))
uns = summary["cat_cols_unseen_over_10pct"]
lines.append(f"Unseen levels: {len(uns)} columns have >10% of production rows with levels unseen in training"
             + (f" ({', '.join(uns)}) - rare/unseen bucketing and an unseen-rate monitor are required." if uns
                else " - rare bucketing is a safeguard rather than a necessity."))
summary["findings"] = lines
for i, l in enumerate(lines, 1):
    print(f"{i}. {l}")

if not LOCAL:
    print("\n", store.put_json(summary, f"{C.ARTIFACTS_PREFIX}/eda_summary.json"))